In [4]:
cd /Users/karolinegriesbach/Documents/Innkeepr/Git/evaluation-and-execution-scripts/

In [5]:
import logging
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from general_functions.datetime_helper import transform_date_to_timestamp_milliseconds
from general_functions.return_workspace_ids import return_workspace_ids
from general_functions.constants import return_api_url
from general_functions.call_api_with_account_id import call_api_with_accountId, send_to_innkeepr_api_paginated

In [ ]:
customer = "More"
from_date = '2026-04-10' #pd.to_datetime(to_date) - pd.DateOffset(days=3)
to_date = '2026-04-16'
#from_date = from_date.strftime('%Y-%m-%d')
goal_name = {
    "More": ["checkout_completed"],
    "ESN": ["checkout_completed"],
    "Whacky Food": ["checkout_completed"],
    "Pendix": ["erwaegung"],
    "LILLYDOO": ["checkout_completed"],
}#, "checkout_started", "product_added_to_cart", "product viewed"]}#,"checkout_completed"] #"checkout_completed", "tagmanagerregistercompleted"
url = return_api_url()
print(f"url = {url}")
account_id = return_workspace_ids()
account_id = [acc["id"] for acc in account_id if acc["name"] == customer]
account_id = account_id[0]

In [7]:
# Tatsächliches Testpaket:
# DACH: Testpaket
# ES: Paquete de prueba
# IT: Pacchetto prova
# FR: Kit d'essai
# NL: Testpakket
testpakete = ["Testpaket", "Paquete de prueba", "Pacchetto prova", "Kit d'essai", "Testpakket"]

# irrelevante Testpakte
# Testpaket-Größe Windeln als Abo-Zusatz:
# DE: LILLYDOO Testpaket Windeln
# ES: Paquete de prueba de pañales LILLYDOO
# IT: Pacchetto prova LILLYDOO
# FR: Échantillon Couches LILLYDOO
# NL: LILLYDOO Testpakket Luiers
# ENGL: LILLYDOO Trial Pack Diapers (SB)
# Testpaket-Größe Windeln als Loyalty Produkt:
# LILLYDOO Trial Pack Diapers (SP)

irrelevante_testpakete = ["LILLYDOO Testpaket Windeln", "Paquete de prueba de pañales LILLYDOO", "Pacchetto prova LILLYDOO", "Échantillon Couches LILLYDOO", "LILLYDOO Testpakket Luiers", "LILLYDOO Trial Pack Diapers (SB)", "LILLYDOO Trial Pack Diapers (SP)"]

In [8]:
conversions=pd.DataFrame()
list_dates = pd.date_range(from_date, to_date).strftime("%Y-%m-%d")
for goal in goal_name[customer]:
    for idate, date in enumerate(list_dates):
        if idate == len(list_dates)-1:
            end_date = list_dates[-1]
        else:
            end_date = list_dates[idate+1]
        print(f"{goal}: Query conversions from {date} to {end_date}")
        content={
            "created": {
                            "$gte": transform_date_to_timestamp_milliseconds(date),
                            "$lte": transform_date_to_timestamp_milliseconds(end_date),
                    },
                    #"name":goal
                        }
        temp=send_to_innkeepr_api_paginated(
            f"{url}/conversions/query",
            account_id,
            content,
            logging
        )
        temp = pd.json_normalize(temp)
        if temp.empty:
            print(".... no conversions for that date")
        print(".... len temp:", len(temp))
        conversions = pd.concat([conversions, temp])
conversions

In [9]:
import json
with open(f'{customer}_conversions_{from_date}.json', 'w') as f:
    json.dump(conversions["sessionId"].tolist(), f, indent=4)


In [10]:
conversions["created"].min(), conversions["created"].max()

In [11]:
conversions["date"] = pd.to_datetime(conversions["created"]).dt.date
print(conversions.shape)
vc = conversions.groupby("date")["name"].value_counts(dropna=False).rename("count").reset_index()
vc.to_csv(f"DataChecks/conversions/conversions_{customer}.csv",index=False)
print(vc['count'].sum())
vc

In [12]:
import ast
def return_product_name(x):
    if x is None:
        return x
    if isinstance(x, str): 
        x = ast.literal_eval(x)
    if isinstance(x, list):
        names = []
        for entry in x:
            if "name" in entry.keys():
                names.append(entry["name"])
        return names
    
conversions["properties.products.name"] = conversions["properties.products"].apply(lambda x: return_product_name(x))
products = conversions["properties.products.name"].explode().drop_duplicates()
products

In [14]:
products[products.isin(testpakete)]